# Class 36: Spectroscopic Fitting: Equivalent Widths and Spectral Line Profiles
## Objective: Understand astronomical spectra and how to fit them

These exercises are based on those in "Lecture 21: Spectroscopic Fitting: Equivalent Widths and Spectral Line Profiles" by Yuan-Sen Ting and available from https://tingyuansen.github.io/coding_essential_for_astronomers/lectures/lecture21-spectroscopic-fitting.html

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
from scipy import signal
from scipy import special
from astropy import units as u
from astropy import constants as const

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib for better-looking plots
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## Section 1: Spectra and Stellar Resolution

The spectral resolution is a measure of how well we can separate two spectral lines. It is the smallest wavelength difference that can be measured and is analogous to angular resolution in imaging. 

The spectral resolution is often written as:

$$
R = \frac{\lambda}{\Delta \lambda}
$$

The instrumental resolution of a spectrograph is commonly described by the **instrumental profile** or **line spread function**. This describes what the instrument would observe for a monochromatic source. The most common functional form is a Gaussian. For an absorption line:

$$
f(\lambda) = 1 - A \exp \left( - \frac{(\lambda - \lambda_0)^2}{2 \sigma^2} \right)
$$

Just as for imaging, the FWHM$ = 2.355\sigma$ for a Gaussian. 

Here is an artificial stellar spectrum with three absorption lines.

In [ ]:
# Create wavelength array (in nanometers)
wavelength = np.linspace(500, 560, 3000)  # 60 nm range, 0.02 nm sampling

# Create continuum with a small slope
continuum = 1.0 - 0.05 * (wavelength - 530) / 30  # small positive slope

# Add absorption lines at specific wavelengths (representing different elements)
# Line 1: Magnesium at 517.3 nm
line1_center = 517.3
line1_depth = 0.15
line1_width = 0.08
absorption1 = line1_depth * np.exp(-0.5 * ((wavelength - line1_center) / line1_width)**2)

# Line 2: Iron at 532.8 nm
line2_center = 532.8
line2_depth = 0.25
line2_width = 0.10
absorption2 = line2_depth * np.exp(-0.5 * ((wavelength - line2_center) / line2_width)**2)

# Line 3: Iron at 543.5 nm
line3_center = 543.5
line3_depth = 0.20
line3_width = 0.09
absorption3 = line3_depth * np.exp(-0.5 * ((wavelength - line3_center) / line3_width)**2)

# Combine: continuum minus absorption
flux = continuum - absorption1 - absorption2 - absorption3

# Add realistic noise
noise_level = 0.005
flux_noisy = flux + np.random.normal(0, noise_level, len(wavelength))

# Plot the spectrum
plt.figure(figsize=(12, 5))
plt.plot(wavelength, flux_noisy, 'k-', linewidth=0.5, alpha=0.6, label='Observed')
plt.plot(wavelength, flux, 'r-', linewidth=1.5, alpha=0.7, label='True (noiseless)')
plt.plot(wavelength, continuum, color='gray', linestyle='--', alpha=0.7, label='Continuum level')  # plot the sloped continuum
plt.xlabel('Wavelength (nm)')
plt.ylabel('Normalized Flux')
plt.title('Stellar Spectrum with Absorption Lines')
plt.ylim([0.6, 1.1])
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Here is an illustration of different spectral resolutions

In [ ]:
# 1. Define spectrograph configurations for the table
configs = {
    'Low-res (imaging spectrograph)': 100,
    'Medium-res (multi-object)': 2000,
    'High-res (echelle)': 50000,
    'Ultra-high-res (laser comb)': 100000
}

# Reference wavelength (Sodium D region)
lambda_ref = 589 * u.nm

print("Spectral Resolution Comparison at λ = 589 nm (Sodium D region):\n")
print(f"{'Configuration':<30} {'R':<10} {'FWHM (nm)':<12}")
print("-" * 55)

for name, R in configs.items():
    delta_lambda = lambda_ref / R
    print(f"{name:<30} {R:<10} {delta_lambda.value:<12.3f}")

# 2. Simulate the Sodium Doublet
# The Sodium D lines are at ~588.995 nm (D2) and ~589.592 nm (D1)
centers = [588.995, 589.592]
wavelength_fine = np.linspace(587, 591, 10000)

# Intrinsic properties
line_width_intrinsic = 0.015  # nm (intrinsic sigma)
amplitudes_intrinsic = [0.6, 0.4] # D2 is usually stronger than D1

# Create intrinsic profile (sum of two Gaussians)
flux_intrinsic = 1.0
for center, amp in zip(centers, amplitudes_intrinsic):
    flux_intrinsic -= amp * np.exp(-0.5 * ((wavelength_fine - center) / line_width_intrinsic)**2)

# 3. Define Resolutions to illustrate
# We'll calculate the FWHM based on the actual R values
resolutions_to_plot = {
    'R = 100,000 (Ultra-High)': 589.0 / 100000,
    'R = 10,000 (Medium-High)': 589.0 / 10000,
    'R = 2,000 (Medium-Low)': 589.0 / 2000,
    'R = 500 (Low)': 589.0 / 500
}

# 4. Plotting
plt.figure(figsize=(12, 10))

for i, (label, fwhm_nm) in enumerate(resolutions_to_plot.items()):
    # Calculate observed sigma based on instrumental FWHM
    sigma_instrument = fwhm_nm / 2.355
    sigma_observed = np.sqrt(line_width_intrinsic**2 + sigma_instrument**2)
    
    # Calculate observed flux (preserving EW by adjusting amplitude)
    # amp_obs = amp_int * (sigma_int / sigma_obs)
    flux_observed = 1.0
    for center, amp_int in zip(centers, amplitudes_intrinsic):
        amp_obs = amp_int * (line_width_intrinsic / sigma_observed)
        flux_observed -= amp_obs * np.exp(-0.5 * ((wavelength_fine - center) / sigma_observed)**2)
    
    # Create Subplot
    plt.subplot(4, 1, i+1)
    
    # Zoom in on the doublet region
    mask = (wavelength_fine >= 588.0) & (wavelength_fine <= 590.5)
    
    plt.plot(wavelength_fine[mask], flux_observed[mask], 'b-', linewidth=1.5, label='Observed Data')
    plt.plot(wavelength_fine[mask], flux_intrinsic[mask], 'r--', linewidth=1, alpha=0.3, label='Intrinsic Truth')
    
    plt.ylabel('Normalized Flux')
    plt.title(f"{label} | FWHM = {fwhm_nm:.3f} nm")
    plt.ylim([0.3, 1.1])
    plt.xlim([588.2, 590.3])
    plt.grid(alpha=0.2)
    
    if i == 0:
        plt.legend(loc='lower right', fontsize=9)
    if i == 3:
        plt.xlabel('Wavelength (nm)')

plt.suptitle('Resolving the Sodium D Doublet: The Effect of Spectral Resolution', fontsize=14, y=0.95)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

**Test your understanding:** You are given a spectrograph with $R=2,000$. Can you distinguish two lines separated by 0.1 nm at a wavelength of 500 nm? Support your answer with a calculation of $\Delta \lambda$.

## Section 2: Broadening Functions

There are three distribution functions that are particularly important in spectroscopy. 

### Thermal Broadening and the Gaussian

A Gaussian is a good description of random thermal velocities. Note that lighter atoms (like Hydrogen) move faster than heavier ones (like Iron) at the same temperature, which produces broader lines.

A Gaussian is also a reasonable approximation to the instrumental line spread function of many spectrographs.  

### Natural or Lorentzian Broadening

The Heisenberg uncertainty principle sets a limit on how narrow lines may be. The Lorentzian profile is:

$$
F(\lambda) = F_c - \frac{A}{1 + \left[ \frac{2 (\lambda - \lambda_0)}{\gamma} \right]^2 }
$$

where $F_c$ is the continuum flux, $A$ is the line amplitude, $\lambda_0$ is the line center$, and $\gamma$ is the Lorentzian FWHM parameter. A Lorentzian has broader wings than a Gaussian.

Pressure broadening also produces a Lorentzian profile. Pressure broadening occurs because deeper layers in stars are at higher pressures and these exhibit broader lines. 

### Voigt Profile

A Voigt profile is the convolution of a Gaussian and Lorentzian. 
This is the most physically accurate description, as it combines both effects. 

$$
V (\lambda) = \int^{\inf}_{-\inf} G(\lambda') L(\lambda - \lambda') d\lambda'
$$

Here is a comparison of the three profiles:

In [ ]:
# Visualize Gaussian, Lorentzian, and Voigt profiles
wavelength = np.linspace(499.85, 500.15, 1000)
continuum = 1.0
amplitude = 0.5
center = 500.0

# Typical stellar line parameters
sigma = 0.020 / 2.355  # Thermal broadening (FWHM = 0.020 nm)
gamma = 0.005  # Pressure broadening (FWHM = 0.005 nm)

# Create profiles directly (without defining functions yet)
# Gaussian
flux_gaussian = continuum - amplitude * np.exp(-0.5 * ((wavelength - center) / sigma)**2)

# Lorentzian
flux_lorentzian = continuum - amplitude / (1 + (2 * (wavelength - center) / gamma)**2)

# Voigt (using scipy voigt_profile function)
from scipy.special import voigt_profile as scipy_voigt
x = wavelength - center
voigt = scipy_voigt(x, sigma, gamma)
voigt = voigt / voigt.max()  # Normalize
flux_voigt = continuum - amplitude * voigt

# Plot comparison
plt.figure(figsize=(10, 6))
plt.plot(wavelength, flux_gaussian, 'c-', linewidth=2, label='Gaussian (thermal only)', alpha=0.8)
plt.plot(wavelength, flux_lorentzian, 'r-', linewidth=2, label='Lorentzian (pressure only)', alpha=0.8)
plt.plot(wavelength, flux_voigt, 'b-', linewidth=2.5, label='Voigt (thermal + pressure)', alpha=0.9)
plt.axhline(y=continuum, color='gray', linestyle='--', alpha=0.5, label='Continuum')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Normalized Flux')
plt.title('Profile Comparison: Voigt Combines Gaussian Core with Lorentzian Wings')
plt.legend()
plt.ylim([0.45, 1.05])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Test your understanding:** Two spectral lines in the same star come from Hydrogen and Iron. Both are at 6,000 K. If the lines are broadened only by thermal motion, which line will have the larger FWHM? Why does atomic mass matter here?

In [ ]:
# Enter your code here

# Section 3: Fitting Voigt Profiles

A Voigt profile is the realistic choice because it combines thermal (core) and pressure (wings) effects. The core decays quickly like a Gaussian, but the wings linger like a Lorentzian.

Here is a six parameter function for the Voight profile of a line: 

In [ ]:
from scipy.special import voigt_profile as scipy_voigt

def voigt_line(wavelength, continuum, amplitude, center, sigma, gamma):
    """
    Voigt absorption line profile (Gaussian convolved with Lorentzian).
    
    Parameters:
    -----------
    wavelength : array
        Wavelength array
    continuum : float
        Continuum flux level
    amplitude : float
        Line depth (absorption below continuum)
    center : float
        Line center wavelength
    sigma : float
        Gaussian width parameter (thermal broadening)
    gamma : float
        Lorentzian FWHM parameter (pressure broadening)
    
    Returns:
    --------
    flux : array
        Flux including absorption line
    """
    # Voigt profile using scipy's voigt_profile function
    x = wavelength - center
    voigt = scipy_voigt(x, sigma, gamma)
    
    # Normalize and scale
    voigt = voigt / voigt.max()
    absorption = amplitude * voigt
    
    return continuum - absorption

# Create synthetic spectrum with a Voigt line

# Wavelength array
wavelength = np.linspace(499.5, 500.5, 200)

# True parameters
true_continuum = 1.0
true_amplitude = 0.35
true_center = 500.0
true_sigma = 0.012  # Thermal broadening (FWHM ~ 0.028 nm)
true_gamma = 0.004  # Pressure broadening (FWHM ~ 0.004 nm)

# Generate noiseless spectrum
flux_true = voigt_line(wavelength, true_continuum, true_amplitude, 
                       true_center, true_sigma, true_gamma)

# Add realistic noise (S/N ~ 100 per pixel)
noise_level = 0.01
flux_observed = flux_true + np.random.normal(0, noise_level, len(wavelength))

# Plot with y-error bars
plt.figure(figsize=(12, 5))
plt.errorbar(wavelength, flux_observed, yerr=noise_level, fmt='ko', markersize=3, 
             alpha=0.5, label='Observed', elinewidth=1, capsize=2)
plt.plot(wavelength, flux_true, 'r-', linewidth=2, label='True (noiseless)')
plt.axhline(y=true_continuum, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Normalized Flux')
plt.title('Synthetic Spectrum for Fitting (Voigt Profile)')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim([0.5, 1.1])
plt.tight_layout()
plt.show()

print(f"True parameters:")
print(f"  Continuum: {true_continuum:.3f}")
print(f"  Amplitude: {true_amplitude:.3f}")
print(f"  Center: {true_center:.4f} nm")
print(f"  Sigma (Gaussian): {true_sigma:.4f} nm")
print(f"  Gamma (Lorentzian): {true_gamma:.4f} nm")

In [ ]:
# Fit Voigt profile to the spectral line

# Initial guesses (could be estimated by eye or from simple measurements)
initial_continuum = 1.0
initial_amplitude = 0.3
initial_center = 500.0
initial_sigma = 0.015  # Guess for thermal broadening
initial_gamma = 0.005  # Guess for pressure broadening

# Bundle initial guesses
p0 = [initial_continuum, initial_amplitude, initial_center, initial_sigma, initial_gamma]

# Perform fit using curve_fit
# curve_fit finds the parameters that minimize chi-squared
popt, pcov = optimize.curve_fit(voigt_line, wavelength, flux_observed, 
                                p0=p0, sigma=noise_level)

# Extract fitted parameters
fit_continuum, fit_amplitude, fit_center, fit_sigma, fit_gamma = popt

# Calculate uncertainties from covariance matrix
perr = np.sqrt(np.diag(pcov))
err_continuum, err_amplitude, err_center, err_sigma, err_gamma = perr

# Generate best-fit model
flux_fit = voigt_line(wavelength, fit_continuum, fit_amplitude, 
                      fit_center, fit_sigma, fit_gamma)

# Calculate residuals
residuals = flux_observed - flux_fit

# Calculate chi-squared
chi_squared = np.sum((residuals / noise_level)**2)
degrees_of_freedom = len(wavelength) - len(popt)
reduced_chi_squared = chi_squared / degrees_of_freedom

# Print results
print("Fitted parameters:")
print(f"  Continuum: {fit_continuum:.4f} ± {err_continuum:.4f}")
print(f"  Amplitude: {fit_amplitude:.4f} ± {err_amplitude:.4f}")
print(f"  Center: {fit_center:.6f} ± {err_center:.6f} nm")
print(f"  Sigma (Gaussian): {fit_sigma:.6f} ± {err_sigma:.6f} nm")
print(f"  Gamma (Lorentzian): {fit_gamma:.6f} ± {err_gamma:.6f} nm")
print(f"\nFit quality:")
print(f"  χ² = {chi_squared:.2f}")
print(f"  Degrees of freedom = {degrees_of_freedom}")
print(f"  Reduced χ² = {reduced_chi_squared:.3f}")
print(f"\nComparison with true values:")
print(f"  Center offset: {(fit_center - true_center)*1000:.3f} pm (picometers)")
print(f"  Sigma offset: {(fit_sigma - true_sigma)*1000:.3f} pm")
print(f"  Gamma offset: {(fit_gamma - true_gamma)*1000:.3f} pm")

# Visualize the fit and residuals
fig, axes = plt.subplots(2, 1, figsize=(12, 8), 
                         gridspec_kw={'height_ratios': [3, 1]})

# Top panel: data and fit with error bars
axes[0].errorbar(wavelength, flux_observed, yerr=noise_level, fmt='ko', 
                markersize=3, alpha=0.5, elinewidth=0.5, capsize=0, 
                label='Observed')
axes[0].plot(wavelength, flux_true, 'r--', linewidth=1.5, alpha=0.7, label='True')
axes[0].plot(wavelength, flux_fit, 'b-', linewidth=2, label='Best fit')
axes[0].axhline(y=fit_continuum, color='gray', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Normalized Flux')
axes[0].set_title(f'Spectral Line Fit (Reduced $\\chi^2$ = {reduced_chi_squared:.3f})')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim([0.5, 1.1])

# Bottom panel: residuals
axes[1].errorbar(wavelength, residuals, yerr=noise_level, fmt='ko', 
                markersize=3, alpha=0.5, elinewidth=0.5, capsize=0)
axes[1].axhline(y=0, color='r', linestyle='-', linewidth=2)
axes[1].axhline(y=noise_level, color='gray', linestyle='--', alpha=0.5)
axes[1].axhline(y=-noise_level, color='gray', linestyle='--', alpha=0.5)
axes[1].fill_between(wavelength, -noise_level, noise_level, alpha=0.2, color='gray')
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('Residuals')
axes[1].set_title(f'Residuals (RMS = {np.std(residuals):.4f})')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Illustrating a Poor Fit (Gaussian vs. Voigt) ---

def gaussian_line(wavelength, continuum, amplitude, center, sigma):
    """Simple Gaussian absorption line for comparison."""
    return continuum - amplitude * np.exp(-0.5 * ((wavelength - center) / sigma)**2)

# Initial Guesses (Trying to fit a Gaussian to our Voigt data)
# We will "underestimate" the width by ignoring the Lorentzian wings
wrong_sigma = 0.015  
wrong_amplitude = 0.40
wrong_center = 500.0

# Generate the poor fit model
flux_poor_fit = gaussian_line(wavelength, true_continuum, wrong_amplitude, 
                              wrong_center, wrong_sigma)

# Calculate Residuals (Data - Model)
residuals = flux_observed - flux_poor_fit

# Plotting the Poor Fit and Residuals
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, 
                               gridspec_kw={'height_ratios': [2, 1]})

# Top Plot: Data and Poor Fit
ax1.errorbar(wavelength, flux_observed, yerr=noise_level, fmt='ko', 
             markersize=3, alpha=0.3, label='Observed Data')
ax1.plot(wavelength, flux_poor_fit, 'r-', linewidth=2, 
         label='Poor Fit (Gaussian - missing wings)')
ax1.set_ylabel('Normalized Flux')
ax1.set_title('Diagnostic of a Poor Fit: Systematic Residuals')
ax1.legend()
ax1.grid(alpha=0.3)

# Bottom Plot: Residuals
ax2.plot(wavelength, residuals, 'ro', markersize=4, alpha=0.6)
ax2.axhline(y=0, color='black', linestyle='--')
ax2.set_ylabel('Residuals (O - C)')
ax2.set_xlabel('Wavelength (nm)')
ax2.set_ylim([-0.1, 0.1])
ax2.grid(alpha=0.3)

# Annotate the residual pattern
ax2.annotate('The "W" pattern indicates\nmissing pressure wings', 
             xy=(500.0, -0.05), xytext=(500.2, -0.08),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1))

plt.tight_layout()
plt.show()

**Test your understanding:** You fit a Gaussian to a spectral line, but the residuals show a clear 'W' shape (the model is too shallow in the wings). Which profile should you try next, and what physical process does this suggest is happening in the star's atmosphere?

In [ ]:
# Enter your code here

## Section 4: Equivalent Width

EW measures the "total absorption" of a line. Imagine a rectangle of height 1 (the continuum) and width W. The area of this rectangle equals the area "missing" from the spectrum due to the absorption line.

In [ ]:
# Visualize equivalent width concept

# Use our fitted line from before
wavelength_plot = wavelength
flux_plot = flux_fit

# Calculate normalized flux (depth below continuum)
depth = 1 - flux_plot / fit_continuum

# Calculate equivalent width by integration
# EW = integral of (1 - F/Fc) dλ
dλ = wavelength_plot[1] - wavelength_plot[0]
equivalent_width = np.sum(depth) * dλ

# For visualization: find the wavelength range where depth > 0.01
mask_line = depth > 0.01
λ_line_min = wavelength_plot[mask_line].min()
λ_line_max = wavelength_plot[mask_line].max()

# Create equivalent rectangle for visualization
# Rectangle width = EW, height = 1.0 (full continuum to zero)
rect_center = fit_center
rect_half_width = equivalent_width / 2

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: show absorption area
axes[0].plot(wavelength_plot, flux_plot, 'b-', linewidth=2, label='Line profile')
axes[0].axhline(y=fit_continuum, color='gray', linestyle='--', alpha=0.5, label='Continuum')
axes[0].fill_between(wavelength_plot, flux_plot, fit_continuum, 
                     where=(depth > 0), alpha=0.3, color='red', 
                     label='Absorbed flux (area = EW)')
axes[0].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('Normalized Flux')
axes[0].set_title('Equivalent Width: Integrated Absorption')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim([0.5, 1.1])

# Right panel: equivalent rectangle
axes[1].plot(wavelength_plot, flux_plot, 'b-', linewidth=2, label='Line profile')
axes[1].axhline(y=fit_continuum, color='gray', linestyle='--', alpha=0.5, label='Continuum')

# Draw equivalent rectangle
rect_λ = [rect_center - rect_half_width, rect_center + rect_half_width,
          rect_center + rect_half_width, rect_center - rect_half_width,
          rect_center - rect_half_width]
rect_flux = [fit_continuum, fit_continuum, 0, 0, fit_continuum]
axes[1].plot(rect_λ, rect_flux, 'r-', linewidth=3, alpha=0.7, 
            label=f'Equivalent rectangle\n(width = {equivalent_width*1000:.2f} pm)')
axes[1].fill_between([rect_center - rect_half_width, rect_center + rect_half_width],
                     0, fit_continuum, alpha=0.2, color='red')

axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('Normalized Flux')
axes[1].set_title('Equivalent Width Concept')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim([-0.1, 1.1])

plt.tight_layout()
plt.show()

print(f"Equivalent width: {equivalent_width*1000:.3f} pm ({equivalent_width:.6f} nm)")
print(f"Line FWHM: {2.355 * fit_sigma*1000:.3f} pm")
print(f"Ratio EW/FWHM: {equivalent_width / (2.355 * fit_sigma):.3f}")

In [ ]:
# Show EW conservation across different resolutions

# Create a sharp intrinsic line - use wider range than plot to avoid edge effects
lambda_conservation = np.linspace(499.0, 501.0, 6000)  # Wider for clean convolution
sigma_intrinsic_cons = 0.010  # Thermal broadening (narrow line)
gamma_intrinsic_cons = 0.002  # Small pressure broadening
flux_intrinsic_cons = voigt_line(lambda_conservation, 1.0, 0.35, 500.0, 
                                 sigma_intrinsic_cons, gamma_intrinsic_cons)

# Calculate intrinsic EW
depth_intrinsic_cons = 1 - flux_intrinsic_cons
d_lambda_cons = lambda_conservation[1] - lambda_conservation[0]
EW_intrinsic_value = np.sum(depth_intrinsic_cons) * d_lambda_cons

# Different instrumental resolutions
resolutions_dict = {
    'R = 100,000': 0.005,  # sigma_instrument in nm
    'R = 50,000': 0.010,
    'R = 20,000': 0.025,
    'R = 10,000': 0.050,
    'R = 5,000': 0.100
}

# Plot and calculate EW for each resolution
fig_cons, axes_cons = plt.subplots(2, 3, figsize=(14, 8))
axes_cons = axes_cons.flatten()

EWs_measured = []

for i, (label, sigma_inst) in enumerate(resolutions_dict.items()):
    # Create Gaussian kernel for convolution
    dlambda_pixel = lambda_conservation[1] - lambda_conservation[0]
    sigma_in_pixels = sigma_inst / dlambda_pixel
    kernel_halfsize = int(5 * sigma_in_pixels)
    kernel_halfsize = max(kernel_halfsize, 3)
    
    # Ensure kernel is not longer than the array (prevents dimension mismatch)
    max_halfsize = len(lambda_conservation) // 2 - 1
    kernel_halfsize = min(kernel_halfsize, max_halfsize)
    
    x_kernel_range = np.arange(-kernel_halfsize, kernel_halfsize + 1)
    kernel_array = np.exp(-0.5 * (x_kernel_range / sigma_in_pixels)**2)
    kernel_array = kernel_array / kernel_array.sum()
    
    # Convolve - use 'same' mode to preserve array size
    flux_convolved = np.convolve(flux_intrinsic_cons, kernel_array, mode='same')
    
    # Calculate EW (only in central region to avoid edge effects)
    mask_central = (lambda_conservation >= 499.7) & (lambda_conservation <= 500.3)
    depth_convolved = 1 - flux_convolved[mask_central]
    EW_measured = np.sum(depth_convolved) * d_lambda_cons
    EWs_measured.append(EW_measured)
    
    # Plot only central region to avoid edge artifacts
    axes_cons[i].plot(lambda_conservation[mask_central], flux_convolved[mask_central], 
                     'b-', linewidth=1.5)
    axes_cons[i].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    axes_cons[i].fill_between(lambda_conservation[mask_central], flux_convolved[mask_central], 
                             1.0, alpha=0.3, color='red')
    axes_cons[i].set_xlabel('Wavelength (nm)')
    axes_cons[i].set_ylabel('Normalized Flux')
    axes_cons[i].set_title(f'{label}\nEW = {EW_measured*1000:.2f} pm')
    axes_cons[i].set_ylim([0.5, 1.05])
    axes_cons[i].set_xlim([499.7, 500.3])
    axes_cons[i].grid(alpha=0.3)

# Summary in last panel
axes_cons[5].axis('off')
axes_cons[5].text(0.1, 0.7, 'Equivalent Width Conservation:', fontsize=14, weight='bold')
axes_cons[5].text(0.1, 0.55, f'Intrinsic EW: {EW_intrinsic_value*1000:.2f} pm', fontsize=12)
axes_cons[5].text(0.1, 0.45, '\nMeasured EW at different R:', fontsize=12)
for j, (label, EW_val) in enumerate(zip(resolutions_dict.keys(), EWs_measured)):
    axes_cons[5].text(0.1, 0.35 - j*0.08, f'  {label}: {EW_val*1000:.2f} pm', fontsize=11)
axes_cons[5].text(0.1, -0.05, f'\nAll measurements agree within ~1%!', fontsize=12, 
            style='italic', color='green')

plt.tight_layout()
plt.show()

print("Equivalent Width conservation:")
print(f"Intrinsic EW: {EW_intrinsic_value*1000:.3f} pm")
for label, EW_val in zip(resolutions_dict.keys(), EWs_measured):
    deviation = (EW_val - EW_intrinsic_value) / EW_intrinsic_value * 100
    print(f"{label:15s}: EW = {EW_val*1000:.3f} pm (deviation: {deviation:+.2f}%)")

In [ ]:
print(resolutions_dict.items())

In [ ]:
for i, (label, sigma_inst) in enumerate(resolutions_dict.items()):
    print(i, label, sigma_inst)

**Test your understanding:** A star is rotating so fast that its spectral lines become broad and shallow. Does the Equivalent Width of these lines increase, decrease, or stay the same? Explain using the PSF analogy from last week.

In [ ]:
# Enter your code here

## Section 5: Measuring Equivalent Width

The best direct approach is to use numerical integration:

$$
EW = \int (1 - F(\lambda)) d\lambda \approx \sum_i (1 - F_i) \Delta \lambda
$$

This is nonparametric, that is it is independent of any assumption about the line shape.

In [ ]:
# Measure EW using direct integration
from scipy.integrate import trapezoid

# Use our fitted line from earlier (fitted with Voigt profile)
wavelength_ew = wavelength
flux_ew = flux_fit
continuum_ew = fit_continuum

# Define integration limits (where line depth > 0.01 of continuum)
# This threshold ensures we capture the line but exclude pure noise
depth_ew = 1 - flux_ew / continuum_ew
mask_integration = depth_ew > 0.01

λ_integrate = wavelength_ew[mask_integration]
depth_integrate = depth_ew[mask_integration]

# Numerical integration using scipy's trapezoid rule (Lecture 16)
EW_measured = trapezoid(depth_integrate, λ_integrate)

# Uncertainty (assuming uniform noise across all pixels)
n_points = len(λ_integrate)
dλ = λ_integrate[1] - λ_integrate[0]
EW_error = noise_level * np.sqrt(n_points) * dλ

print("Equivalent Width Measurement:")
print(f"  EW = {EW_measured:.6f} ± {EW_error:.6f} nm")

## Section 6: Chemical Abundances

The ultimate goal of stellar spectroscopy is often to determine the chemical "makeup" of a star. We do this by measuring how much light is missing from a spectral line.

### 1. The EW-Abundance Connection
For weak (unsaturated) lines, there is a simple linear relationship between the **Equivalent Width (EW)** and the number of atoms of that element present in the stellar atmosphere:

$$\text{EW} \propto N_{\text{atoms}}$$

* **Key Insight:** If you double the number of atoms, you double the amount of absorbed light (the EW). This linear regime makes calculating abundances straightforward.

### 2. Stellar Abundance Notation $[X/H]$
Because stellar abundances span many orders of magnitude, astronomers use a logarithmic scale relative to the Sun. The bracket notation $[X/H]$ represents the ratio of element $X$ to Hydrogen:

$$[X/H] = \log_{10}\left(\frac{N_X/N_H}{\text{star}}\right) - \log_{10}\left(\frac{N_X/N_H}{\text{Sun}}\right)$$

### 3. How to Interpret the Numbers
Since this is a base-10 log scale, each change of **1.0** represents a **factor of 10** difference in chemistry:

* **$[X/H] = 0$**: Exactly the same as the Sun (**Solar abundance**).
* **$[X/H] = +1$**: 10 times more abundant than the Sun (**Metal-rich**).
* **$[X/H] = -1$**: 10 times less abundant than the Sun (**Metal-poor**).
* **$[Fe/H] = -2$**: 100 times less iron than the Sun (Typical for very old stars in our Galaxy).

In [ ]:
# Demonstrate EW-abundance relationship for weak lines

# Simulate lines with different abundances
abundances_relative = np.array([0.3, 0.5, 1.0, 2.0, 3.0])  # Relative to solar

# Fixed line properties
wavelength_demo = np.linspace(499.8, 500.2, 400)
center_demo = 500.0
sigma_demo = 0.012  # Base line width
gamma_demo = 0.003  # Small pressure broadening

# For weak, unsaturated lines: amplitude ∝ abundance (linear regime)
# These lines have optically thin cores - each atom contributes independently
amplitudes = 0.15 * abundances_relative  # Linear with abundance
sigmas = np.full_like(abundances_relative, sigma_demo)  # Width stays constant

# Calculate EW for each abundance using scipy.integrate
from scipy.integrate import trapezoid

EWs = []
for amplitude, sigma in zip(amplitudes, sigmas):
    flux = voigt_line(wavelength_demo, 1.0, amplitude, center_demo, sigma, gamma_demo)
    depth = 1 - flux
    EW = trapezoid(depth[depth > 0], wavelength_demo[depth > 0])
    EWs.append(EW)
EWs = np.array(EWs)

# Plot: show how lines change with abundance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Lines at different abundances
for abundance, amplitude, sigma in zip(abundances_relative, amplitudes, sigmas):
    flux = voigt_line(wavelength_demo, 1.0, amplitude, center_demo, sigma, gamma_demo)
    axes[0].plot(wavelength_demo, flux, linewidth=2, 
                 label=f'[X/H] = {np.log10(abundance):+.1f}')
axes[0].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('Normalized Flux')
axes[0].set_title('Weak Lines: Depth Increases with Abundance\n(Width Remains Constant)')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim([0.5, 1.05])

# Right: EW vs abundance (linear relationship)
axes[1].plot(abundances_relative, EWs * 1000, 'bo-', 
            linewidth=2, markersize=8, label='Measured EW')
# Fit a line to show linearity
fit_slope = np.polyfit(abundances_relative, EWs * 1000, 1)[0]
axes[1].plot(abundances_relative, fit_slope * abundances_relative, 
            'r--', linewidth=2, alpha=0.7, label=f'Linear fit (slope = {fit_slope:.2f} pm)')
axes[1].set_xlabel('Abundance (relative to solar)')
axes[1].set_ylabel('Equivalent Width (pm)')
axes[1].set_title('EW ∝ Abundance (Linear Relationship)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("EW-Abundance Relationship (Weak Lines):\n")
for abundance, EW in zip(abundances_relative, EWs):
    log_abundance = np.log10(abundance)
    print(f"  [X/H] = {log_abundance:+.1f} ({abundance:.1f}×solar) → EW = {EW*1000:.2f} pm")
print(f"\nThe linear fit confirms: EW increases by {fit_slope:.2f} pm per 1×solar abundance")

**Test your understanding:** A star has an iron abundance of [Fe/H]=−2. How many iron atoms does this star have compared to the Sun? (Is it 2 times less, 10 times less, or 100 times less?)

In [ ]:
# Enter your code here

## Section 7: Stellar Rotation

As a star rotates, one hemisphere moves toward us (blueshifted) while the other moves away (redshifted). This opposing motion "smears" the absorption lines, making them broader and shallower.

Because of the star's orientation, we only measure the **projected** rotation velocity:

$$v_{obs} = v \sin i$$

The measured $v \sin i$ is always a **lower limit** to the star's true equatorial velocity. A star viewed pole-on ($i = 0^\circ$) will show no rotational broadening at all, even if it is spinning rapidly. This is the same geometric degeneracy we encountered with exoplanet masses ($M \sin i$).

In [ ]:
# Simulate and fit rotationally broadened line

# Create intrinsic line (narrow, from non-rotating star)
wavelength_rot = np.linspace(499.7, 500.3, 600)
center_rot = 500.0 * u.nm

# Thermal width (from temperature ~6000K)
# For typical atom: v_thermal ~ 2 km/s
v_thermal = 2.0 * u.km / u.s

# Convert thermal velocity to wavelength broadening using Doppler formula
# Δλ/λ = v/c → Δλ = λ × v/c
# FWHM in wavelength space
delta_lambda_thermal = center_rot * v_thermal / const.c
sigma_thermal = (delta_lambda_thermal / 2.355).to(u.nm).value  # Convert FWHM to sigma

# Rotational velocity
vsini = 25.0 * u.km / u.s  # moderate rotation

# Calculate rotational broadening in wavelength
# Δλ_rot / λ = v_rot / c
delta_lambda_rot = center_rot * vsini / const.c
sigma_rot = (delta_lambda_rot / 2.355).to(u.nm).value

# Combined width: add thermal and rotational widths in quadrature
# This assumes both broadenings are approximately Gaussian
sigma_total = np.sqrt(sigma_thermal**2 + sigma_rot**2)

# Create intrinsic (non-rotating) line profile using Gaussian
amplitude_rot = 0.40
flux_thermal_only = 1.0 - amplitude_rot * np.exp(-0.5 * ((wavelength_rot - center_rot.value) / sigma_thermal)**2)

# Create rotationally broadened line (broader Gaussian)

# IMPORTANT: Scale amplitude to conserve equivalent width (total absorbed flux)
# For Gaussian: EW = amplitude × sigma × sqrt(2π), so amplitude ∝ 1/sigma
amplitude_broadened = amplitude_rot * (sigma_thermal / sigma_total)
flux_rotated = 1.0 - amplitude_broadened * np.exp(-0.5 * ((wavelength_rot - center_rot.value) / sigma_total)**2)

# Add noise
noise_level_rot = 0.008
flux_rotated_noisy = flux_rotated + np.random.normal(0, noise_level_rot, 
                                                      len(wavelength_rot))

# Plot the line
plt.figure(figsize=(12, 5))
plt.errorbar(wavelength_rot, flux_rotated_noisy, yerr=noise_level_rot,
            fmt='ko', markersize=2, alpha=0.5, elinewidth=0.5, capsize=0,
            label='Observed (rotating star)')
plt.plot(wavelength_rot, flux_thermal_only, 'b--', linewidth=2, alpha=0.7, 
         label=f'Non-rotating (v sin i = 0 km/s)')
plt.plot(wavelength_rot, flux_rotated, 'r-', linewidth=2, 
         label=f'Rotating (v sin i = {vsini.value} km/s)')

plt.xlabel('Wavelength (nm)')
plt.ylabel('Normalized Flux')
plt.title('Effect of Stellar Rotation on Line Profile')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim([0.5, 1.05])
plt.tight_layout()
plt.show()

print(f"True parameters:")
print(f"  v_thermal = {v_thermal}")
print(f"  v sin(i) = {vsini}")
print(f"  σ_thermal = {sigma_thermal*1000:.3f} pm")
print(f"  σ_rotation = {sigma_rot*1000:.3f} pm")
print(f"  σ_total = {sigma_total*1000:.3f} pm")

### Section 8: Continuum Fitting

Just as crowded fields in imaging require simultaneous PSF fitting to separate stars, spectral lines often overlap or sit on a sloped background. To analyze them, we must first "flatten" the spectrum through continuum normalization.

The continuum is the "true" brightness of the star's surface. If your continuum fit is too high, your lines will look too deep; if it is too low, you will underestimate the chemical abundances. We typically use iterative sigma-clipping to identify true continuum points by automatically rejecting pixels that belong to absorption lines. Once identified, we fit a smooth function (like a polynomial) to these points and divide the entire spectrum by this fit to set the background level to 1.0.

In [ ]:
# Demonstrate local continuum normalization for individual line fitting

# Create a local region around a single line with sloped continuum
wavelength_local = np.linspace(499.5, 500.5, 200)  # 1 nm region
line_center = 500.0
line_width = 0.015

# Local continuum with a slope (common in real spectra)
# Slope might come from instrumental response, blaze function, etc.
continuum_local = 1.05 - 0.08 * (wavelength_local - 500.0)  # 8% slope per nm

# Add an absorption line
line_amplitude = 0.30
line_profile = line_amplitude * np.exp(-0.5 * ((wavelength_local - line_center) / line_width)**2)
flux_with_slope = continuum_local - line_profile

# Add noise
noise_local = 0.008
flux_observed_local = flux_with_slope + np.random.normal(0, noise_local, len(wavelength_local))

# Plot the problem
plt.figure(figsize=(12, 5))
plt.errorbar(wavelength_local, flux_observed_local, yerr=noise_local,
            fmt='ko', markersize=3, alpha=0.5, elinewidth=0.5, capsize=0,
            label='Observed spectrum')
plt.plot(wavelength_local, continuum_local, 'r--', linewidth=2, 
         label='True continuum (sloped)')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Flux (arbitrary units)')
plt.title('Local Spectrum Around a Line: Sloped Continuum')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Challenge: Local continuum has slope")
print(f"  Continuum at 499.5 nm: {continuum_local[0]:.3f}")
print(f"  Continuum at 500.5 nm: {continuum_local[-1]:.3f}")
print(f"  Slope: {(continuum_local[-1] - continuum_local[0]):.3f} ({(continuum_local[-1] - continuum_local[0]) / continuum_local.mean() * 100:.1f}%)")

In [ ]:
# Perform iterative continuum fitting with sigma-clipping

# Initialize: start by assuming all points might be continuum
continuum_mask = np.ones(len(flux_observed_local), dtype=bool)

# Iteratively: fit continuum, then clip outliers, then refit
print("Iterative continuum fitting with sigma-clipping:")
for iteration in range(3):
    # Select currently unmasked points as continuum candidates
    wavelength_continuum = wavelength_local[continuum_mask]
    flux_continuum = flux_observed_local[continuum_mask]
    
    # Fit a linear function: continuum(λ) = slope × λ + intercept
    # We use np.polyfit(x, y, degree) which fits a polynomial
    # degree=1 gives a line: y = coef[0]*x + coef[1]
    coefficients = np.polyfit(wavelength_continuum, flux_continuum, deg=1)
    continuum_fitted = np.polyval(coefficients, wavelength_local)
    
    # Calculate residuals: observed - fitted continuum
    residuals = flux_observed_local - continuum_fitted
    residual_std = np.std(residuals[continuum_mask])
    
    # Update mask: keep points within 2-sigma of the fitted continuum
    # Points significantly BELOW the fit are likely absorption lines
    continuum_mask = (residuals > -2.0 * residual_std)
    
    print(f"  Iteration {iteration+1}: {continuum_mask.sum()} continuum pixels, "
          f"{(~continuum_mask).sum()} line pixels")

# Final continuum fit parameters
print(f"\nFinal fitted continuum:")
print(f"  Slope: {coefficients[0]*1000:.3f} per nm")
print(f"  Intercept: {coefficients[1]:.3f}")

# Normalize spectrum by dividing by fitted continuum
flux_normalized = flux_observed_local / continuum_fitted

# Visualize the normalization process
fig, axes = plt.subplots(2, 1, figsize=(12, 9), gridspec_kw={'height_ratios': [1, 1]})

# Top panel: show original spectrum with continuum fit
axes[0].errorbar(wavelength_local, flux_observed_local, yerr=noise_local,
                fmt='ko', markersize=3, alpha=0.4, elinewidth=0.5, capsize=0,
                label='Observed spectrum')
axes[0].plot(wavelength_local[continuum_mask], flux_observed_local[continuum_mask], 
            'go', markersize=5, alpha=0.8,
            label='Identified continuum points')
axes[0].plot(wavelength_local, continuum_local, 'r--', linewidth=2, alpha=0.7, 
            label='True continuum')
axes[0].plot(wavelength_local, continuum_fitted, 'b-', linewidth=2, 
            label='Fitted continuum')
axes[0].set_ylabel('Flux (arbitrary units)')
axes[0].set_title('Original Spectrum: Iterative Continuum Fit with Sigma-Clipping')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Bottom panel: show normalized spectrum
axes[1].errorbar(wavelength_local, flux_normalized, yerr=noise_local/continuum_fitted,
                fmt='ko', markersize=3, alpha=0.4, elinewidth=0.5, capsize=0,
                label='Normalized spectrum')
axes[1].axhline(y=1.0, color='r', linestyle='--', linewidth=2, alpha=0.7, 
               label='Continuum = 1')
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('Normalized Flux')
axes[1].set_title('After Normalization: Ready for Line Fitting')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim([0.6, 1.1])

plt.tight_layout()
plt.show()

print(f"\nNormalized spectrum statistics:")
print(f"  Mean continuum level: {np.mean(flux_normalized[continuum_mask]):.4f}")
print(f"  Continuum scatter: {np.std(flux_normalized[continuum_mask]):.3f}")

**Test your understanding:** Imagine you wrote a continuum normalization code and you accidentally included a few pixels from the bottom of an absorption line in your fit. How will this error affect your measured line depth (will it be too deep or too shallow)?

## Section 9: Fitting Blended Lines

In [ ]:
# Create spectrum with blended lines

# Wavelength array (high resolution around blend)
wavelength_blend = np.linspace(499.85, 500.15, 600)

# Create a close doublet (two lines separated by only 0.05 nm)
line1_center_blend = 500.00
line2_center_blend = 500.05  # Only 0.05 nm away!

# Both lines have similar properties
line1_amplitude_blend = 0.35
line2_amplitude_blend = 0.25  # Slightly weaker
line1_sigma_blend = 0.012
line2_sigma_blend = 0.010

# Create blended spectrum
flux_blend = 1.0 * np.ones_like(wavelength_blend)
absorption1 = line1_amplitude_blend * np.exp(
    -0.5 * ((wavelength_blend - line1_center_blend) / line1_sigma_blend)**2
)
absorption2 = line2_amplitude_blend * np.exp(
    -0.5 * ((wavelength_blend - line2_center_blend) / line2_sigma_blend)**2
)
flux_blend -= (absorption1 + absorption2)

# Add noise
noise_blend = 0.006
flux_blend_noisy = flux_blend + np.random.normal(0, noise_blend, len(wavelength_blend))

# Plot to show the blend
plt.figure(figsize=(12, 5))
plt.errorbar(wavelength_blend, flux_blend_noisy, yerr=noise_blend,
            fmt='ko', markersize=2, alpha=0.5, elinewidth=0.5, capsize=0,
            label='Observed (blended)')
plt.plot(wavelength_blend, flux_blend, 'k-', linewidth=2, label='True (both lines)')

# Show individual components
flux_line1_only = 1.0 - absorption1
flux_line2_only = 1.0 - absorption2
plt.plot(wavelength_blend, flux_line1_only, 'r--', linewidth=2, alpha=0.7, 
         label=f'Line 1 alone ({line1_center_blend:.2f} nm)')
plt.plot(wavelength_blend, flux_line2_only, 'b--', linewidth=2, alpha=0.7, 
         label=f'Line 2 alone ({line2_center_blend:.2f} nm)')

plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Normalized Flux')
plt.title('Blended Spectral Lines (Separation = 0.05 nm)')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim([0.4, 1.1])
plt.tight_layout()
plt.show()

print("True parameters:")
print(f"  Line 1: center = {line1_center_blend:.4f} nm, amplitude = {line1_amplitude_blend:.3f}")
print(f"  Line 2: center = {line2_center_blend:.4f} nm, amplitude = {line2_amplitude_blend:.3f}")
print(f"  Separation: {(line2_center_blend - line1_center_blend)*1000:.1f} pm")
print(f"  Separation in FWHM units: {(line2_center_blend - line1_center_blend) / (2.355*line1_sigma_blend):.2f}")

# Fit blended lines simultaneously
def two_line_model(wavelength, continuum, amp1, center1, sigma1, gamma1, 
                   amp2, center2, sigma2, gamma2):
    """Model with two Voigt absorption lines (for fitting blends)"""
    # Start with continuum, then subtract both absorption components
    line1 = voigt_line(wavelength, continuum, amp1, center1, sigma1, gamma1)
    line2 = voigt_line(wavelength, continuum, amp2, center2, sigma2, gamma2)
    # Combined absorption: continuum minus both absorptions
    flux = continuum - (continuum - line1) - (continuum - line2)
    return flux

# Initial guesses (crucial for blends!)
# Need reasonable starting points or fit may fail
p0_blend = [
    1.0,  # continuum
    0.35, 500.00, 0.012, 0.002,  # line 1: amp, center, sigma, gamma
    0.25, 500.05, 0.010, 0.002   # line 2: amp, center, sigma, gamma
]

# Fit
popt_blend, pcov_blend = optimize.curve_fit(two_line_model, wavelength_blend, 
                                            flux_blend_noisy, p0=p0_blend)

# Extract parameters
(fit_cont_blend, 
 fit_amp1_blend, fit_center1_blend, fit_sigma1_blend, fit_gamma1_blend,
 fit_amp2_blend, fit_center2_blend, fit_sigma2_blend, fit_gamma2_blend) = popt_blend

perr_blend = np.sqrt(np.diag(pcov_blend))

# Generate model components
flux_fit_blend = two_line_model(wavelength_blend, *popt_blend)
flux_comp1_blend = voigt_line(wavelength_blend, fit_cont_blend, fit_amp1_blend,
                               fit_center1_blend, fit_sigma1_blend, fit_gamma1_blend)
flux_comp2_blend = voigt_line(wavelength_blend, fit_cont_blend, fit_amp2_blend,
                               fit_center2_blend, fit_sigma2_blend, fit_gamma2_blend)

# Calculate residuals
residuals_blend = flux_blend_noisy - flux_fit_blend
chi_squared_blend = np.sum((residuals_blend / noise_blend)**2)
dof_blend = len(wavelength_blend) - len(popt_blend)
reduced_chi_squared_blend = chi_squared_blend / dof_blend

print("Fitted parameters:\n")
print(f"Line 1:")
print(f"  Center: {fit_center1_blend:.6f} ± {perr_blend[2]:.6f} nm "
      f"(true: {line1_center_blend:.4f})")
print(f"  Amplitude: {fit_amp1_blend:.4f} ± {perr_blend[1]:.4f} "
      f"(true: {line1_amplitude_blend:.3f})")
print(f"  σ: {fit_sigma1_blend:.6f} nm, γ: {fit_gamma1_blend:.6f} nm")
# Calculate EW by integrating the fitted Voigt component
depth1 = 1 - flux_comp1_blend / fit_cont_blend
EW1 = np.trapezoid(depth1[depth1 > 0], wavelength_blend[depth1 > 0])
print(f"  EW: {EW1*1000:.3f} pm")

print(f"\nLine 2:")
print(f"  Center: {fit_center2_blend:.6f} ± {perr_blend[6]:.6f} nm "
      f"(true: {line2_center_blend:.4f})")
print(f"  Amplitude: {fit_amp2_blend:.4f} ± {perr_blend[5]:.4f} "
      f"(true: {line2_amplitude_blend:.3f})")
print(f"  σ: {fit_sigma2_blend:.6f} nm, γ: {fit_gamma2_blend:.6f} nm")
# Calculate EW by integrating the fitted Voigt component
depth2 = 1 - flux_comp2_blend / fit_cont_blend
EW2 = np.trapezoid(depth2[depth2 > 0], wavelength_blend[depth2 > 0])
print(f"  EW: {EW2*1000:.3f} pm")

print(f"\nFit quality:")
print(f"  Reduced χ² = {reduced_chi_squared_blend:.3f}")

# Visualize blend fitting results
fig, axes = plt.subplots(2, 1, figsize=(12, 9), 
                         gridspec_kw={'height_ratios': [3, 1]})

# Top panel: data, fit, and components
axes[0].errorbar(wavelength_blend, flux_blend_noisy, yerr=noise_blend,
                fmt='ko', markersize=2, alpha=0.4, elinewidth=0.5, capsize=0,
                label='Observed')
axes[0].plot(wavelength_blend, flux_fit_blend, 'b-', linewidth=2.5, 
            label='Best fit (both lines)')
axes[0].plot(wavelength_blend, flux_comp1_blend, 'r--', linewidth=2, alpha=0.7,
            label=f'Line 1 component ({fit_center1_blend:.4f} nm)')
axes[0].plot(wavelength_blend, flux_comp2_blend, 'g--', linewidth=2, alpha=0.7,
            label=f'Line 2 component ({fit_center2_blend:.4f} nm)')

axes[0].axhline(y=fit_cont_blend, color='gray', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Normalized Flux')
axes[0].set_title(f'Blended Line Fit - Successfully Deblended! (Reduced $\\chi^2$ = {reduced_chi_squared_blend:.3f})')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim([0.4, 1.1])

# Bottom panel: residuals
axes[1].errorbar(wavelength_blend, residuals_blend, yerr=noise_blend,
                fmt='ko', markersize=2, alpha=0.5, elinewidth=0.5, capsize=0)
axes[1].axhline(y=0, color='b', linestyle='-', linewidth=2)
axes[1].fill_between(wavelength_blend, -noise_blend, noise_blend, 
                     alpha=0.2, color='gray')
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('Residuals')
axes[1].set_title(f'Residuals (RMS = {np.std(residuals_blend):.5f})')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Solutions to In-Class Exercises

### Section 1 Solution

To determine if the lines are resolvable, calculate the instrumental resolution Δλ:

$$
\Delta \lambda = \frac{\lambda}{R} = \frac{500\,\mathrm{nm}}{2,000} = 0.25\,\mathrm{nm}
$$

Result: Since the instrument's resolution (0.25 nm) is larger than the line separation (0.1 nm), you cannot distinguish the two lineshey will appear blurred together as a single feature.

### Section 2 Solution

Result: The Hydrogen line will have the larger FWHM.

Reasoning: Thermal broadening is driven by the average speed of atoms, which is inversely proportional to their mass ($v_{th} \propto \sqrt{T/m}$). Because Hydrogen is much lighter than Iron, its atoms move significantly faster at the same temperature (6,000 K), resulting in a larger Doppler spread and a broader spectral line.

### Section 3 Solution

Result: You should try fitting a Voigt profile next.

Reasoning: A "W" shape in the residuals indicates that your Gaussian model is failing to account for broad "wings" in the spectral line (the Gaussian decays too quickly, leaving data points in the wings poorly fit). Physically, this suggests that pressure broadening (collisional broadening) is occurring in the star's atmosphere.

While a Gaussian only accounts for thermal motion, the Voigt profile is a convolution of a Gaussian and a Lorentzian. The Lorentzian component provides the extended wings necessary to model the effects of high pressure or density in the stellar atmosphere.

### Section 4 Solution

Reasoning: Using the PSF analogy from last week, a star’s total flux remains constant regardless of whether the "seeing" is good or bad. The the light is simply redistributed over a larger area. Similarly, rotational broadening redistributes the absorption over a wider wavelength range, which makes the line shallower, but the total amount of light that is removed (the area of the line) remains unchanged. Therefore, the EW is a robust measurement that is independent of how much the line is "smeared" by rotation.

### Section 6 Solution

Result: The star has 100 times less iron than the Sun.

Reasoning: The abundance notation [Fe/H] is a base-10 logarithmic scale. A value of −2 means the ratio of iron to hydrogen is $10^{-2}$ or 1/100) times the solar ratio.

[Fe/H]=0: Solar abundance ($10^0=1$)

[Fe/H]=−1: 10 times less than the Sun ($10^{-1} =0.1$)

[Fe/H]=−2: 100 times less than the Sun ($10^{-2}=0.01$)